In [0]:
#display(dbutils.fs.ls("/Volumes/marathos/default/raw/"))

#%restart_python

In [0]:
#place it in a _setup notebook inside the explorations folder

#need to run this start of each cell one time for the notebook to work, dont like having filepaths, so I did this

#import sys

#PROJECT_ROOT = "/Workspace/Users/<email>/<projectname(root)/<pyproject location> if pyproject is in root this last one isnt needed"

#if PROJECT_ROOT not in sys.path:
    #sys.path.insert(0, PROJECT_ROOT)

In [0]:
%run ../explorations/_setup

In [0]:
%skip
from importlib import reload
import transformations.bronze.ingest_bronze as bronze_module
reload(bronze_module)

In [0]:
from utils.helpers import get_table
from pyspark.sql import functions as F
from transformations.bronze.ingest_bronze import ingest_bronze

ingest_bronze(spark)

In [0]:
df = get_table("marathos.bronze.raw_results")

In [0]:
print(f"Number of rows:    {df.count()}")
print(f"Number of columns: {len(df.columns)}")

In [0]:
#bronze layer
df.printSchema()

In [0]:
df.limit(5).display()

In [0]:
df.describe().display()

In [0]:
null_counts = df.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)
    ).alias(c)
    for c in df.columns
])

null_counts.display()

In [0]:
unique_events = df.select("event_name").distinct().count()
print(f"Number of unique events: {unique_events}")

In [0]:
df.select("event_name").distinct().limit(20).display()

In [0]:
df.groupBy("event_distance_raw") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.select("athlete_performance_raw") \
  .distinct() \
  .limit(20) \
  .display()

In [0]:
df.withColumn(
    "birth_year", F.col("athlete_year_of_birth").cast("double").cast("int")
).withColumn(
    "event_year", F.col("year_of_event").cast("int")
).withColumn(
    "age", F.col("event_year") - F.col("birth_year")
).groupBy("age") \
 .count() \
 .orderBy("age") \
 .filter(F.col("age").isNotNull()) \
 .display()

In [0]:
df.groupBy("athlete_country") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(20) \
  .display()

In [0]:
df.groupBy("athlete_gender") \
  .count() \
  .orderBy(F.desc("count")) \
  .display()

In [0]:
df.groupBy("year_of_event") \
  .count() \
  .orderBy("year_of_event") \
  .display()

some checks for bronze layer above.

In [0]:
df_bronze.groupBy("event_distance_raw").count().orderBy(F.desc("count")).limit(20).display()

In [0]:
df_bronze.filter(
    F.col("event_distance_raw").like("%h%")
).select("event_distance_raw", "athlete_performance_raw") \
 .limit(10) \
 .display()

In [0]:
df_bronze.filter(
    F.col("event_distance_raw") == "12h"
).select("event_distance_raw", "athlete_performance_raw", "athlete_id_raw") \
 .limit(10) \
 .display()

In [0]:
df_bronze.groupBy("athlete_id_raw") \
  .agg(
    F.countDistinct("athlete_performance_raw").alias("unique_performances"),
    F.countDistinct("athlete_country").alias("unique_countries"),
    F.count("*").alias("total_rows")
  ) \
  .filter(F.col("unique_performances") > 1) \
  .orderBy(F.desc("total_rows")) \
  .limit(10) \
  .display()

In [0]:
df_bronze.groupBy(
    "athlete_country",
    "athlete_year_of_birth", 
    "athlete_gender",
    "athlete_club"
) \
.count() \
.orderBy(F.desc("count")) \
.limit(10) \
.display()

In [0]:
#check if event_name + year_of_event is unique
df_bronze.groupBy("event_name", "year_of_event") \
  .count() \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .display()

In [0]:
#need to check if there are multiple event_dates for the same event_name + year_of_event
df_bronze.groupBy("event_name", "year_of_event") \
  .agg(F.countDistinct("event_dates").alias("unique_dates")) \
  .filter(F.col("unique_dates") > 1) \
  .orderBy(F.desc("unique_dates")) \
  .limit(10) \
  .display()